In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split , GridSearchCV

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet , BayesianRidge , SGDRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor , AdaBoostRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.kernel_ridge import KernelRidge
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

from keras.models import Sequential
from keras.layers import Dense, Input
from keras.optimizers import Adam

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score

In [2]:
df = pd.read_excel('C:/Users/yasmi/Downloads/data_JC.xls', sheet_name="Feuil1")

df.head()

,A(MPa),B(MPa),n,Umin,FUmin,UFmax,Fmax,a1(pente1)x0,a2(pente2)x2,ptflexion
0,10,90,0.2,0.472616,132.514948,9.406141,1828.338135,-41.048167,128.671351,5.846848
1,45,90,0.2,0.472787,151.169610,9.403861,2582.698486,-40.003432,145.262882,4.290309
2,80,90,0.2,0.472771,194.113349,9.417878,3347.173828,-109.336813,186.211358,3.454615
3,10,145,0.2,0.472679,126.031387,9.416237,2814.554688,30.753123,205.591396,5.586041
4,45,145,0.2,0.472779,167.995440,9.408020,3576.553467,-90.515814,254.893251,5.236132


In [4]:
#X= df.drop(columns = ['A(MPa)' , 'B(MPa)' , 'n'], axis=1)

X_B = df.drop(columns = ['B(MPa)'], axis=1)
y_B = df[["B(MPa)"]]
X2_train, X2_test, y2_train, y2_test = train_test_split(X_B, y_B, test_size=0.15, random_state=42)

In [5]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(max_iter=10000),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'SGD': SGDRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'LightGBM': LGBMRegressor(verbose=-1),
    'SVR': SVR(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'MLP': MLPRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'CatBoost': CatBoostRegressor(verbose=0, random_seed=42),
    'Bayesian Ridge': BayesianRidge(),
    'Kernel Ridge': KernelRidge(),
    'KNN': KNeighborsRegressor()
}

In [6]:
param_grids = {
    'Ridge': {
        'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
        'solver': ['auto', 'saga', 'lsqr'],
        'max_iter': [1000, 5000, 10000]
    },
    'Lasso': {
        'alpha': [1e-4, 1e-3, 1e-2, 0.1],
        'max_iter': [1000, 5000, 10000]
    },
    'ElasticNet': {
        'alpha': [1e-4, 1e-3, 1e-2, 0.1],
        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
        'max_iter': [1000, 5000, 10000]
    },
    'SGD': {
        'alpha': [0.0001, 0.001, 0.01],
        'penalty': ['l2', 'l1', 'elasticnet'],
        'learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'],
        'eta0': [0.001, 0.01, 0.1],
        'max_iter': [10000 , 20000]
    },
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10]
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7]
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 6, 10]
    },
    'LightGBM': {
        'n_estimators': [200, 500, 1000],
        'learning_rate': [0.01, 0.05],
        'num_leaves': [15, 31, 63],
        'min_data_in_leaf': [1, 10, 20]
    },
    'SVR': {
        'C': [0.1, 1, 10, 100],
        'epsilon': [0.001, 0.01, 0.1],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'Decision Tree': {
        'max_depth': [5, 10, 20],
        'min_samples_split': [2, 5, 10]
    },
    'MLP': {
        'hidden_layer_sizes': [(100,), (200,100), (300,200,100)],
        'learning_rate_init': [0.001, 0.01],
        'max_iter': [5000, 10000, 20000]
    },
    'AdaBoost': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1.0]
    },
    'CatBoost': {
        'iterations': [500, 1000],
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1]
    },
    'Bayesian Ridge': {
        'alpha_1': [1e-6, 1e-5],
        'alpha_2': [1e-6, 1e-5],
        'lambda_1': [1e-6, 1e-5],
        'lambda_2': [1e-6, 1e-5]
    },
    'Kernel Ridge': {
        'alpha': [0.01, 0.1, 1.0],
        'kernel': ['linear', 'rbf'],
        'gamma': [0.01, 0.1, 1.0]
    },
    'KNN': {
        'n_neighbors': [3, 5, 6, 7],
        'weights': ['uniform', 'distance']
    }
}

In [7]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def run_grid_search_minmax(model_name, model, X_train, y_train, param_grid, scoring='neg_mean_squared_error', cv=5):
    print(f"🔍 Tuning {model_name}...")
    y_train = np.ravel(y_train)
    pipe = Pipeline([
        ('scaler', MinMaxScaler()),
        ('model', model)
    ])
    search = GridSearchCV(pipe, {f'model__{k}': v for k, v in param_grid.items()}, 
                          scoring=scoring, cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    print(f"✅ Best params for {model_name}: {search.best_params_}")
    return search

In [16]:
from sklearn.metrics import r2_score

best_models = {}
y1_train = np.ravel(y2_train)
y1_test = np.ravel(y2_test)
for name, model in models.items():
    if name in param_grids:
        search = run_grid_search_minmax(name, model, X2_train, y2_train, param_grids[name])
        best_models[name] = search.best_estimator_
        
        # Predict on test set
        y_pred = search.best_estimator_.predict(X2_test)
        r2 = r2_score(y2_test, y_pred)
        print(f"R² score for {name} with best hyperparameters: {r2:.4f}")
        
    else:
        print(f" No hyperparameter grid for {name}, using default model.")
        model.fit(X2_train, y2_train)
        best_models[name] = model
        
        y_pred = model.predict(X2_test)
        r2 = r2_score(y2_test, y_pred)
        print(f"R² score for {name} with default hyperparameters: {r2:.4f}")

 No hyperparameter grid for Linear Regression, using default model.
R² score for Linear Regression with default hyperparameters: 0.9184
🔍 Tuning Ridge...
✅ Best params for Ridge: {'model__alpha': 0.1, 'model__max_iter': 10000, 'model__solver': 'saga'}
R² score for Ridge with best hyperparameters: 0.9038
🔍 Tuning Lasso...
✅ Best params for Lasso: {'model__alpha': 0.1, 'model__max_iter': 1000}
R² score for Lasso with best hyperparameters: 0.9204
🔍 Tuning ElasticNet...
✅ Best params for ElasticNet: {'model__alpha': 0.01, 'model__l1_ratio': 0.1, 'model__max_iter': 1000}
R² score for ElasticNet with best hyperparameters: 0.8829
🔍 Tuning SGD...
✅ Best params for SGD: {'model__alpha': 0.01, 'model__eta0': 0.001, 'model__learning_rate': 'adaptive', 'model__max_iter': 10000, 'model__penalty': 'elasticnet'}
R² score for SGD with best hyperparameters: 0.8615
🔍 Tuning Random Forest...
✅ Best params for Random Forest: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 

C:\Users\yasmi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Best params for SVR: {'model__C': 100, 'model__epsilon': 0.001, 'model__gamma': 'scale', 'model__kernel': 'linear'}
R² score for SVR with best hyperparameters: 0.8903
🔍 Tuning Decision Tree...
✅ Best params for Decision Tree: {'model__max_depth': 5, 'model__min_samples_split': 10}
R² score for Decision Tree with best hyperparameters: 0.2193
🔍 Tuning MLP...


C:\Users\yasmi\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(


✅ Best params for MLP: {'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.001, 'model__max_iter': 5000}
R² score for MLP with best hyperparameters: 0.7752
🔍 Tuning AdaBoost...
✅ Best params for AdaBoost: {'model__learning_rate': 0.01, 'model__n_estimators': 50}
R² score for AdaBoost with best hyperparameters: 1.0000
🔍 Tuning CatBoost...
✅ Best params for CatBoost: {'model__depth': 6, 'model__iterations': 1000, 'model__learning_rate': 0.05}
R² score for CatBoost with best hyperparameters: 0.6822
🔍 Tuning Bayesian Ridge...
✅ Best params for Bayesian Ridge: {'model__alpha_1': 1e-06, 'model__alpha_2': 1e-05, 'model__lambda_1': 1e-05, 'model__lambda_2': 1e-06}
R² score for Bayesian Ridge with best hyperparameters: 0.9389
🔍 Tuning Kernel Ridge...
✅ Best params for Kernel Ridge: {'model__alpha': 0.01, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
R² score for Kernel Ridge with best hyperparameters: 0.8316
🔍 Tuning KNN...
✅ Best params for KNN: {'model__n_neighbors': 3, 'mode

In [8]:
def run_grid_search_stand(model_name, model, X_train, y_train, param_grid, scoring='neg_mean_squared_error', cv=5):
    print(f"🔍 Tuning {model_name}...")
    y_train = np.ravel(y_train)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    search = GridSearchCV(pipe, {f'model__{k}': v for k, v in param_grid.items()}, 
                          scoring=scoring, cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    print(f"✅ Best params for {model_name}: {search.best_params_}")
    return search

In [18]:
best_models = {}
y1_train = np.ravel(y2_train)
y1_test = np.ravel(y2_test)
for name, model in models.items():
    if name in param_grids:
        search = run_grid_search_stand(name, model, X2_train, y2_train, param_grids[name])
        best_models[name] = search.best_estimator_
        
        y_pred = search.best_estimator_.predict(X2_test)
        r2 = r2_score(y2_test, y_pred)
        print(f"R² score for {name} with best hyperparameters: {r2:.4f}")
        
    else:
        print(f" No hyperparameter grid for {name}, using default model.")
        model.fit(X2_train, y2_train)
        best_models[name] = model
        
        y_pred = model.predict(X2_test)
        r2 = r2_score(y2_test, y_pred)
        print(f"R² score for {name} with default hyperparameters: {r2:.4f}")

 No hyperparameter grid for Linear Regression, using default model.
R² score for Linear Regression with default hyperparameters: 0.9184
🔍 Tuning Ridge...
✅ Best params for Ridge: {'model__alpha': 10, 'model__max_iter': 1000, 'model__solver': 'saga'}
R² score for Ridge with best hyperparameters: 0.8255
🔍 Tuning Lasso...
✅ Best params for Lasso: {'model__alpha': 0.1, 'model__max_iter': 1000}
R² score for Lasso with best hyperparameters: 0.9575
🔍 Tuning ElasticNet...
✅ Best params for ElasticNet: {'model__alpha': 0.1, 'model__l1_ratio': 0.1, 'model__max_iter': 1000}
R² score for ElasticNet with best hyperparameters: 0.9043
🔍 Tuning SGD...
✅ Best params for SGD: {'model__alpha': 0.01, 'model__eta0': 0.001, 'model__learning_rate': 'invscaling', 'model__max_iter': 10000, 'model__penalty': 'l2'}
R² score for SGD with best hyperparameters: 0.9180
🔍 Tuning Random Forest...
✅ Best params for Random Forest: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 300}
R² s

C:\Users\yasmi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Best params for SVR: {'model__C': 100, 'model__epsilon': 0.001, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
R² score for SVR with best hyperparameters: 0.6552
🔍 Tuning Decision Tree...
✅ Best params for Decision Tree: {'model__max_depth': 5, 'model__min_samples_split': 10}
R² score for Decision Tree with best hyperparameters: 0.2193
🔍 Tuning MLP...
✅ Best params for MLP: {'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.001, 'model__max_iter': 10000}
R² score for MLP with best hyperparameters: 0.0996
🔍 Tuning AdaBoost...
✅ Best params for AdaBoost: {'model__learning_rate': 0.01, 'model__n_estimators': 50}
R² score for AdaBoost with best hyperparameters: 1.0000
🔍 Tuning CatBoost...
✅ Best params for CatBoost: {'model__depth': 6, 'model__iterations': 1000, 'model__learning_rate': 0.05}
R² score for CatBoost with best hyperparameters: 0.6822
🔍 Tuning Bayesian Ridge...
✅ Best params for Bayesian Ridge: {'model__alpha_1': 1e-06, 'model__alpha_2': 1e-05, 'model__lam

In [10]:
best_models = {}
results = []

y1_train = np.ravel(y2_train)
y1_test = np.ravel(y2_test)

for name, model in models.items():
    if name in param_grids:
        search = run_grid_search_stand(name, model, X2_train, y2_train, param_grids[name])
        best_models[name] = search.best_estimator_
        
        y_pred = search.best_estimator_.predict(X2_test)
        r2 = r2_score(y2_test, y_pred)
        print(f"R² score for {name} with best hyperparameters: {r2:.4f}")
        
        # Store results
        results.append({
            "Model": name,
            "Test R²": r2
        })
        
    else:
        print(f"⚠️ No hyperparameter grid for {name}, using default model.")
        model.fit(X2_train, y2_train)
        best_models[name] = model
        
        y_pred = model.predict(X2_test)
        r2 = r2_score(y2_test, y_pred)
        print(f"R² score for {name} with default hyperparameters: {r2:.4f}")
        
        # Store results (no tuning info)
        results.append({
            "Model": name,
            "Test R²": r2
        })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)
print("\n📊 Results Table:")
print(results_df)

⚠️ No hyperparameter grid for Linear Regression, using default model.
R² score for Linear Regression with default hyperparameters: 0.9184
🔍 Tuning Ridge...
✅ Best params for Ridge: {'model__alpha': 10, 'model__max_iter': 5000, 'model__solver': 'saga'}
R² score for Ridge with best hyperparameters: 0.8255
🔍 Tuning Lasso...
✅ Best params for Lasso: {'model__alpha': 0.1, 'model__max_iter': 1000}
R² score for Lasso with best hyperparameters: 0.9575
🔍 Tuning ElasticNet...
✅ Best params for ElasticNet: {'model__alpha': 0.1, 'model__l1_ratio': 0.1, 'model__max_iter': 1000}
R² score for ElasticNet with best hyperparameters: 0.9043
🔍 Tuning SGD...
✅ Best params for SGD: {'model__alpha': 0.01, 'model__eta0': 0.001, 'model__learning_rate': 'invscaling', 'model__max_iter': 10000, 'model__penalty': 'l2'}
R² score for SGD with best hyperparameters: 0.9180
🔍 Tuning Random Forest...
✅ Best params for Random Forest: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 300}
R²

C:\Users\yasmi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Best params for SVR: {'model__C': 100, 'model__epsilon': 0.001, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
R² score for SVR with best hyperparameters: 0.6552
🔍 Tuning Decision Tree...
✅ Best params for Decision Tree: {'model__max_depth': 5, 'model__min_samples_split': 10}
R² score for Decision Tree with best hyperparameters: 0.2193
🔍 Tuning MLP...
✅ Best params for MLP: {'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.001, 'model__max_iter': 10000}
R² score for MLP with best hyperparameters: 0.0996
🔍 Tuning AdaBoost...
✅ Best params for AdaBoost: {'model__learning_rate': 0.01, 'model__n_estimators': 50}
R² score for AdaBoost with best hyperparameters: 1.0000
🔍 Tuning CatBoost...
✅ Best params for CatBoost: {'model__depth': 6, 'model__iterations': 1000, 'model__learning_rate': 0.05}
R² score for CatBoost with best hyperparameters: 0.6822
🔍 Tuning Bayesian Ridge...
✅ Best params for Bayesian Ridge: {'model__alpha_1': 1e-06, 'model__alpha_2': 1e-05, 'model__lam

In [11]:
best_models = {}
results = []

y1_train = np.ravel(y2_train)
y1_test = np.ravel(y2_test)

for name, model in models.items():
    if name in param_grids:
        search = run_grid_search_minmax(name, model, X2_train, y2_train, param_grids[name])
        best_models[name] = search.best_estimator_
        
        y_pred = search.best_estimator_.predict(X2_test)
        r2 = r2_score(y2_test, y_pred)
        print(f"R² score for {name} with best hyperparameters: {r2:.4f}")
        
        # Store results
        results.append({
            "Model": name,
            "Test R²": r2
        })
        
    else:
        print(f"⚠️ No hyperparameter grid for {name}, using default model.")
        model.fit(X2_train, y2_train)
        best_models[name] = model
        
        y_pred = model.predict(X2_test)
        r2 = r2_score(y2_test, y_pred)
        print(f"R² score for {name} with default hyperparameters: {r2:.4f}")
        
        # Store results (no tuning info)
        results.append({
            "Model": name,
            "Test R²": r2
        })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)
print("\n📊 Results Table:")
print(results_df)

⚠️ No hyperparameter grid for Linear Regression, using default model.
R² score for Linear Regression with default hyperparameters: 0.9184
🔍 Tuning Ridge...
✅ Best params for Ridge: {'model__alpha': 0.1, 'model__max_iter': 5000, 'model__solver': 'saga'}
R² score for Ridge with best hyperparameters: 0.9038
🔍 Tuning Lasso...
✅ Best params for Lasso: {'model__alpha': 0.1, 'model__max_iter': 1000}
R² score for Lasso with best hyperparameters: 0.9204
🔍 Tuning ElasticNet...
✅ Best params for ElasticNet: {'model__alpha': 0.01, 'model__l1_ratio': 0.1, 'model__max_iter': 1000}
R² score for ElasticNet with best hyperparameters: 0.8829
🔍 Tuning SGD...
✅ Best params for SGD: {'model__alpha': 0.01, 'model__eta0': 0.001, 'model__learning_rate': 'adaptive', 'model__max_iter': 10000, 'model__penalty': 'elasticnet'}
R² score for SGD with best hyperparameters: 0.8615
🔍 Tuning Random Forest...
✅ Best params for Random Forest: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators':

C:\Users\yasmi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Best params for SVR: {'model__C': 100, 'model__epsilon': 0.001, 'model__gamma': 'scale', 'model__kernel': 'linear'}
R² score for SVR with best hyperparameters: 0.8903
🔍 Tuning Decision Tree...
✅ Best params for Decision Tree: {'model__max_depth': 5, 'model__min_samples_split': 10}
R² score for Decision Tree with best hyperparameters: 0.2193
🔍 Tuning MLP...


C:\Users\yasmi\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(


✅ Best params for MLP: {'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.001, 'model__max_iter': 5000}
R² score for MLP with best hyperparameters: 0.7752
🔍 Tuning AdaBoost...
✅ Best params for AdaBoost: {'model__learning_rate': 0.01, 'model__n_estimators': 50}
R² score for AdaBoost with best hyperparameters: 1.0000
🔍 Tuning CatBoost...
✅ Best params for CatBoost: {'model__depth': 6, 'model__iterations': 1000, 'model__learning_rate': 0.05}
R² score for CatBoost with best hyperparameters: 0.6822
🔍 Tuning Bayesian Ridge...
✅ Best params for Bayesian Ridge: {'model__alpha_1': 1e-06, 'model__alpha_2': 1e-05, 'model__lambda_1': 1e-05, 'model__lambda_2': 1e-06}
R² score for Bayesian Ridge with best hyperparameters: 0.9389
🔍 Tuning Kernel Ridge...
✅ Best params for Kernel Ridge: {'model__alpha': 0.01, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
R² score for Kernel Ridge with best hyperparameters: 0.8316
🔍 Tuning KNN...
✅ Best params for KNN: {'model__n_neighbors': 3, 'mode